# Linguistic bigram frequencies in ED

This notebook counts how often each bigram of the **linguistic bigram tokenizer** is used when the ED labels are encoded. The vocabulary was learned from IAM text only; ED labels are only encoded here, nothing is trained.

It answers:

1. How much of the encoded text is covered by bigrams at all?
2. Is bigram use spread out or dominated by a few bigrams? Are some never used?
3. Do the training and validation parts of a fold use bigrams in the same way?

**Counting rule:** only bigram tokens the encoder actually emits are counted (the encoder scans left to right and takes a pair whenever it is in the vocabulary). This is not a count of every overlapping letter pair. Case, spaces and punctuation are kept; every recording counts, so repeated sentences count repeatedly.

WD and WI contain the same 2,550 recordings, so their **pooled** counts are identical. Only their train/validation splits differ.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import Markdown, display

from loguru import logger
from tva.ed_analysis import js_divergence, load_records, load_tokenizer, show, show_table, token_counts

logger.disable('tva')  # hide routine 'tokenizer loaded' messages

# ---- Settings: change these, then run all cells ----
TOKENIZER = 'linguistic_bigram_tva'  # original TVA bigram trained on IAM text
# Alternatives: 'linguistic_bigram_iam_frequency' (letter pairs ranked by IAM count),
#               'handwriting_bigram' (for comparison)
SPLITS = ['wd', 'wi']
FOLD = 0
TOP_N = 30   # 145 shows every vocabulary bigram

tokenizer = load_tokenizer(TOKENIZER)
bigrams = [token for token in tokenizer.vocab if len(token) == 2]
print(f'{TOKENIZER}: {len(bigrams)} bigrams, {tokenizer.size} classes including blank')

In [ ]:
def count(records):
    tokens = token_counts(tokenizer, (tokenizer.encode(record['label']) for record in records))
    return {'records': len(records), 'tokens': sum(tokens.values()),
            'bigrams': Counter({t: n for t, n in tokens.items() if len(t) == 2})}


results = {split: {part: count(load_records(split, FOLD, part)) for part in ('train', 'val')}
           for split in SPLITS}
first = results[SPLITS[0]]
pooled = first['train']['bigrams'] + first['val']['bigrams']
pooled_total = sum(pooled.values())
ranked = sorted(bigrams, key=lambda t: (-pooled[t], t))
unused = [t for t in ranked if pooled[t] == 0]
print(f'Pooled ED (train + validation): {pooled_total:,} bigram tokens out of '
      f"{first['train']['tokens'] + first['val']['tokens']:,} tokens")

## 1. Summary

- **Bigram share** — the percentage of all emitted tokens that are bigrams (the rest are single characters).
- **Bigrams used** — how many of the vocabulary's bigrams occur at least once.
- **Divergence** — Jensen–Shannon divergence between the training and validation bigram frequencies of the fold: **0 = identical**, **1 = nothing in common**.
- **Concentration** — how many of the most frequent bigrams are needed to cover 50%, 80% and 95% of all bigram tokens. Small numbers mean a few bigrams dominate.

In [ ]:
rows = []
for split in SPLITS:
    for part in ('train', 'val'):
        r = results[split][part]
        n = sum(r['bigrams'].values())
        rows.append({'Split': split.upper(), 'Part': part, 'Sentences': r['records'],
                     'All tokens': f"{r['tokens']:,}", 'Bigram tokens': f'{n:,}',
                     'Bigram share (%)': f"{100 * n / r['tokens']:.1f}",
                     'Bigrams used': f"{len(r['bigrams'])} of {len(bigrams)}"})
    rows.append({'Split': split.upper(), 'Part': 'train vs val', 'Sentences': '', 'All tokens': '',
                 'Bigram tokens': '', 'Bigram share (%)': '',
                 'Bigrams used': f"divergence {js_divergence(results[split]['train']['bigrams'], results[split]['val']['bigrams']):.4f}"})
show_table(rows)


def needed(share):
    covered = 0
    for rank, token in enumerate(ranked, 1):
        covered += pooled[token]
        if covered >= share * pooled_total:
            return rank


coverage = {share: needed(share) for share in (0.5, 0.8, 0.95)}
top10 = 100 * sum(pooled[t] for t in ranked[:10]) / pooled_total
show_table([{'Top 10 bigrams cover (%)': f'{top10:.1f}',
             'Bigrams for 50%': coverage[0.5], 'Bigrams for 80%': coverage[0.8],
             'Bigrams for 95%': coverage[0.95], 'Never used': f'{len(unused)} of {len(bigrams)}'}])
print('Never-used bigrams:', ', '.join(show(t) for t in unused) or '-')

## 2. Most frequent bigrams

`Share` and `Cumulative` use all pooled bigram tokens. The split columns give each bigram's share within that fold's training or validation bigram tokens. Spaces are visible inside quotes, e.g. `'e '` is *e* followed by a space.

In [ ]:
rows, cumulative = [], 0
for rank, token in enumerate(ranked[:TOP_N], 1):
    cumulative += pooled[token]
    row = {'Rank': rank, 'Bigram': show(token), 'Count': pooled[token],
           'Share (%)': f'{100 * pooled[token] / pooled_total:.2f}',
           'Cumulative (%)': f'{100 * cumulative / pooled_total:.1f}'}
    for split in SPLITS:
        for part in ('train', 'val'):
            c = results[split][part]['bigrams']
            row[f'{split.upper()} {part} (%)'] = f'{100 * c[token] / (sum(c.values()) or 1):.2f}'
    rows.append(row)
show_table(rows)

## 3. Training versus validation

For the top bigrams, each bar is the bigram's share within that part of the fold. Bars of similar length mean training and validation need that bigram equally often; large differences mean the validation sentences use it more or less than the training sentences.

In [ ]:
top = ranked[:TOP_N]
fig, axes = plt.subplots(1, len(SPLITS), figsize=(6 * len(SPLITS), 0.3 * len(top) + 1.5),
                         sharey=True, squeeze=False)
for ax, split in zip(axes[0], SPLITS):
    for k, part in enumerate(('train', 'val')):
        c = results[split][part]['bigrams']
        total = sum(c.values()) or 1
        ax.barh([i + (k - 0.5) * 0.4 for i in range(len(top))], [100 * c[t] / total for t in top],
                height=0.4, label='training' if part == 'train' else 'validation')
    ax.set_yticks(range(len(top)), [show(t) for t in top])
    ax.set_title(f'{split.upper()} fold {FOLD}')
    ax.set_xlabel('Share of bigram tokens (%)')
    ax.legend()
axes[0, 0].invert_yaxis()
plt.tight_layout()
plt.show()

## 4. How concentrated is bigram use?

Left: count of every vocabulary bigram, from most to least frequent. Right: the cumulative share. A curve that rises steeply and then flattens means a few bigrams carry most of the usage and many bigrams are rare.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.bar(range(1, len(ranked) + 1), [pooled[t] for t in ranked], width=1.0)
left.set_xlabel('Bigram rank')
left.set_ylabel('Count in ED (train + validation)')
left.set_title('Count per bigram')

shares, running = [], 0
for token in ranked:
    running += pooled[token]
    shares.append(100 * running / pooled_total)
right.plot(range(1, len(ranked) + 1), shares)
for share, rank in coverage.items():
    right.axhline(100 * share, color='grey', lw=0.6, ls='--')
    right.annotate(f'{int(100 * share)}% at {rank}', (rank, 100 * share), textcoords='offset points',
                   xytext=(5, -12))
right.set_xlabel('Number of most frequent bigrams')
right.set_ylabel('Cumulative share of bigram tokens (%)')
right.set_title('Cumulative coverage')
plt.tight_layout()
plt.show()

## 5. Summary in words

In [ ]:
share =100 * pooled_total / (first['train']['tokens'] + first['val']['tokens'])
lines = [
    f'- Bigrams make up **{share:.1f}%** of all tokens when ED is encoded; the rest are single characters.',
    f'- **{len(bigrams) - len(unused)} of {len(bigrams)}** vocabulary bigrams occur in ED; {len(unused)} are never used.',
    f'- Use is concentrated: the 10 most frequent bigrams cover **{top10:.1f}%** of bigram tokens, '
    f'and **{coverage[0.8]}** bigrams cover 80%.',
]
for split in SPLITS:
    value = js_divergence(results[split]['train']['bigrams'], results[split]['val']['bigrams'])
    lines.append(f'- {split.upper()} fold {FOLD}: training and validation bigram frequencies differ by a divergence of '
                 f'**{value:.4f}** (0 = identical).')
display(Markdown('\n'.join(lines)))